In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
requirements = "/content/drive/MyDrive/pythonprojects_2/final_year_project/3D_project/requirements.txt"
!pip install -r "{requirements}"
import torch
#check if cuda is available
print(f"cuda is available: {torch.cuda.is_available()}")

print("Diffusion mode requirements installed ✅")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 21.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of rembg to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of cupy-cuda12x to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 21.3 MB/s eta 0:0

cuda is available: True
Diffusion mode requirements installed ✅


In [4]:
# To run the processing script
import os
%cd "/content/drive/MyDrive/pythonprojects_2/final_year_project/3D_project"
rawdata_path = f"/content/drive/MyDrive/pythonprojects_2/final_year_project/3D_project/output_baseline_d_s/hotdog/"

def get_name_from_path(path):
    return os.path.basename(os.path.normpath(path))
filename = get_name_from_path(rawdata_path)

/content/drive/MyDrive/pythonprojects_2/final_year_project/3D_project


In [ ]:
import os

# Always get the original base name from rawdata_path to avoid double-prefixing if the cell is run multiple times
current_base_filename = os.path.basename(os.path.normpath(rawdata_path))

# Define the new prefixed filename for the Gaussian Splatting scene
# This will result in 'baseline_hotdog_d_s' based on user request
prefixed_filename = f"baseline_{current_base_filename}_d_s"

# Construct the full destination path in Google Drive for the new scene
drive_scene_path = f"/content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting/input_dataset/{prefixed_filename}"

# Create the main scene directory (e.g., .../input_dataset/baseline_hotdog_d_s/)
!mkdir -p "{drive_scene_path}"

# Create the 'input' subdirectory inside the scene directory
# This is where Gaussian Splatting expects the initial image data for COLMAP
!mkdir -p "{drive_scene_path}/input"

# Copy the *contents* of `rawdata_path` (e.g., all files and subdirectories within `hotdog/`)
# into the newly created `input` directory.
# The wildcard `*` is used to copy the contents of the directory, not the directory itself.
!cp -r "{rawdata_path}"* "{drive_scene_path}/input/"

# Update the global `filename` variable to reflect the new prefixed name
# This ensures subsequent cells use the correct filename (e.g., 'baseline_hotdog_d_s')
filename = prefixed_filename

In [5]:
print("starting the Gaussian Splatting Process")
print("⏳ Installing submodules locally (this might take a moment)...")
!pip install -q plyfile
%cd /content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting
import os
import shutil
import sys
import torch

print("⚔️  STEP 1: REMOVING MEMORY HOGS...")
!pip uninstall -y tensorflow > /dev/null 2>&1
!pip uninstall -y tensorflow-probability > /dev/null 2>&1
print("✅ TensorFlow removed. GPU memory is clean.")

repo_path = "/content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting"

# 👇 THE PIPELINE FIX: Copy submodules to local SSD and compile them natively
print("⚙️ Compiling CUDA Extensions natively on local SSD...")
!rm -rf /content/submodules_local
!cp -r "{repo_path}/submodules" /content/submodules_local

# Destroy any corrupted caches that came from Drive
!rm -rf /content/submodules_local/diff-gaussian-rasterization/build
!rm -rf /content/submodules_local/diff-gaussian-rasterization/*.egg-info
!rm -rf /content/submodules_local/simple-knn/build
!rm -rf /content/submodules_local/simple-knn/*.egg-info

# Install from the local NVMe drive
!pip install -q /content/submodules_local/diff-gaussian-rasterization
!pip install -q /content/submodules_local/simple-knn
print("✅ Submodules Compilation Complete!")

!sudo apt-get install -y ffmpeg
!sudo apt-get update
!sudo apt-get install -y colmap
!colmap -h
!sudo apt install imagemagick -y

print("Import installation for Gaussian Splatting done✅.")

starting the Gaussian Splatting Process
⏳ Installing submodules locally (this might take a moment)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.6 MB/s eta 0:00:00
/content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting
⚔️  STEP 1: REMOVING MEMORY HOGS...
✅ TensorFlow removed. GPU memory is clean.
⚙️ Compiling CUDA Extensions natively on local SSD...
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
✅ Submodules Compilation Complete!
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InR

In [ ]:
import os
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
!pip install pycolmap
!sudo apt-get install -y xvfb

drive_path = f"/content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting/input_dataset/{filename}"
local_path = f"/content/local_workspace/{filename}"

print(f"🚀 Step 1: Transferring {filename} dataset from Google Drive to Local NVMe SSD...")
!mkdir -p "{local_path}"

# 👇 THE FIX: We name the local folder 'input' instead of 'images'
!rm -rf "{local_path}/input"

# We copy the Drive 'images' folder INSIDE the local folder and rename it to 'input'
!cp -r "{drive_path}/input" "{local_path}/input"

print("🧹 Step 2: Cleaning local workspace (Sparse/Distorted)...")
!rm -rf "{local_path}/sparse"
!rm -rf "{local_path}/distorted"
!rm -f "{local_path}/database.db" "{local_path}/database.db-shm" "{local_path}/database.db-wal"

print("⚡ Step 3: Executing AI Pipeline at maximum speed...")
%cd /content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting

# 👇 THE FIX: You no longer need the --images flag!
# Option A: The Original Colmap tool (wrapped with xvfb-run to provide a virtual display)
!xvfb-run -a python convert.py -s "{local_path}"

# Option B: The Exhaustive AI (SuperPoint + LightGlue)
# !xvfb-run -a python convert_ai.py --source_path "{local_path}"

# Option C: The Optimized AI (NetVLAD + LightGlue)
# !xvfb-run -a python convert_ai_v2.py --source_path "{local_path}"

print("💾 Step 4: Transferring finished 3D model back to Google Drive...")
!cp -r "{local_path}/distorted" "{drive_path}/"
!cp -r "{local_path}/sparse" "{drive_path}/"
!cp "{local_path}/database.db" "{drive_path}/"

print(f"✅✅ Process Complete for {filename}!")

Streaming output truncated to the last 5000 lines.
  22  6.844332e+01    1.15e-01    4.03e+02   7.54e+00   6.35e-01  6.02e+04        1    4.87e-03    7.66e-02
  23  6.833249e+01    1.11e-01    3.92e+02   7.49e+00   6.37e-01  6.14e+04        1    4.19e-03    8.08e-02
  24  6.822555e+01    1.07e-01    3.82e+02   7.44e+00   6.39e-01  6.28e+04        1    3.52e-03    8.43e-02
  25  6.812243e+01    1.03e-01    3.72e+02   7.38e+00   6.41e-01  6.42e+04        1    3.15e-03    8.75e-02
  26  6.802309e+01    9.93e-02    3.62e+02   7.33e+00   6.42e-01  6.57e+04        1    3.12e-03    9.07e-02
  27  6.792748e+01    9.56e-02    3.52e+02   7.28e+00   6.44e-01  6.73e+04        1    3.06e-03    9.38e-02
  28  6.783556e+01    9.19e-02    3.43e+02   7.22e+00   6.45e-01  6.90e+04        1    4.30e-03    9.81e-02
  29  6.774730e+01    8.83e-02    3.33e+02   7.17e+00   6.47e-01  7.08e+04        1    3.56e-03    1.02e-01
  30  6.766266e+01    8.46e-02    3.24e+02   7.11e+00   6.48e-01  7.27e+04        1  

In [ ]:
# 👇 THE PATH FIX: Pull from input_dataset, not the raw output
DRIVE_COLMAP_DATA = f"{repo_path}/input_dataset/{filename}"
LOCAL_INPUT = f"/content/local_workspace/{filename}"
LOCAL_OUTPUT = f"/content/local_workspace/{filename}_final_run"
FINAL_DRIVE_DESTINATION = f"{repo_path}/output/{filename}_final_run"

print("📦 STAGING COLMAP DATA ON LOCAL SSD...")
# Clean the local slate and copy the exact folders train.py needs
!rm -rf "{LOCAL_INPUT}"
!mkdir -p "{LOCAL_INPUT}"
!cp -r "{DRIVE_COLMAP_DATA}/input" "{LOCAL_INPUT}/images"
!cp -r "{DRIVE_COLMAP_DATA}/sparse" "{LOCAL_INPUT}/"

print("\n🔍 SANITY CHECK: Verifying scene structure...")
!ls -la "{LOCAL_INPUT}/sparse/0"
print(f"\n🚀 STARTING TRAINING PIPELINE (100% LOCAL I/O)...")
%cd {repo_path}
!python train.py -s "{LOCAL_INPUT}" -m "{LOCAL_OUTPUT}" --eval

# --- SECURE THE RENDER ---
print("\n💾 TRANSFERRING FINAL 3D MODEL TO GOOGLE DRIVE...")
!cp -r "{LOCAL_OUTPUT}" "{repo_path}/output/"

print(f"\n✅✅✅ PIPELINE FINISHED! Model saved permanently to: {FINAL_DRIVE_DESTINATION}")

📦 STAGING COLMAP DATA ON LOCAL SSD...

🔍 SANITY CHECK: Verifying scene structure...
total 768
drwx------ 2 root root   4096 May  5 18:10 .
drwx------ 3 root root   4096 May  5 18:10 ..
-rw------- 1 root root     64 May  5 18:10 cameras.bin
-rw------- 1 root root 670113 May  5 18:10 images.bin
-rw------- 1 root root 102013 May  5 18:10 points3D.bin

🚀 STARTING TRAINING PIPELINE (100% LOCAL I/O)...
/content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting
Optimizing /content/local_workspace/baseline_hotdog_d_s_final_run
Output folder: /content/local_workspace/baseline_hotdog_d_s_final_run [05/05 18:11:11]
------------LLFF HOLD------------- [05/05 18:11:11]
Reading camera 21/21 [05/05 18:11:11]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [05/05 18:11:11]
Loading Training Cameras [05/05 18:11:11]
Loading Test Cameras [05/05 18:11:12]
Number of points at initialisation :  1247 [05/05 18:11:12]
Training progress:  23% 7000/30000 [03

In [10]:
new_filename = "baseline_hotdog_d_s"

In [12]:
import os
%cd /content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting

LOCAL_INPUT = f"/content/local_workspace/{new_filename}"
DRIVE_COLMAP_DATA = f"/content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting/input_dataset/{new_filename}"

# Empty the local workspace to ensure a clean slate, then restage
print("🧹 Cleaning local workspace...")
!rm -rf "{LOCAL_INPUT}"

print("📦 Re-staging dataset to local workspace from Google Drive...")
!mkdir -p "{LOCAL_INPUT}"
!cp -r "{DRIVE_COLMAP_DATA}/input" "{LOCAL_INPUT}/images"
!cp -r "{DRIVE_COLMAP_DATA}/sparse" "{LOCAL_INPUT}/"
print("✅ Re-staging complete!")

# Step 1: Render the held-out views
!python render.py \
    -m "/content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting/output/{new_filename}_final_run" \
    -s "{LOCAL_INPUT}" \
    --skip_train

# Step 2: Calculate SSIM and LPIPS metrics
!python metrics.py \
    -m "/content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting/output/{new_filename}_final_run"

/content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting
🧹 Cleaning local workspace...
📦 Re-staging dataset to local workspace from Google Drive...
✅ Re-staging complete!
Looking for config file in /content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting/output/baseline_hotdog_d_s_final_run/cfg_args
Config file found: /content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting/output/baseline_hotdog_d_s_final_run/cfg_args
Rendering /content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting/output/baseline_hotdog_d_s_final_run
Loading trained model at iteration 30000 [09/05 09:04:32]
------------LLFF HOLD------------- [09/05 09:04:32]
Reading camera 21/21 [09/05 09:04:32]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [09/05 09:04:32]
Loading Training Cameras [09/05 09:04:32]
Loading Test Cameras [09/05 09:04:32]
Rendering progress: 100% 3/3 [00:00<00:00,  3.22it/s]

Scene